In [1]:
import pandas as pd
import yfinance as yf
import duckdb
import re
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [2]:
import logging

# Konfiguration des Loggers
logging.basicConfig(
    filename='errors.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [3]:
class YahooClient:
    def __init__(self, ticker: str):
        self.ticker = ticker
        self._yf = yf.Ticker(ticker)
    
    def identification(self):
        name = self._yf.get_info()["shortName"]
        isin = self._yf.get_isin()
        return name
    
    def get_fx(self):
         return yf.download(self.ticker, period="20y")

    def unit_info(self):
        info = self._yf.get_info()
        market_currency = info.get("currency")
        financial_currency = info.get("financialCurrency")
        if not financial_currency:
            financial_currency = market_currency
        return {"price_history": market_currency, 
                "balance_sheet": financial_currency, 
                "income_statement": financial_currency,
                "cashflow": financial_currency,
                "mutualfund_holders": "percent", 
                "institutional_holders": "percent", 
                "major_holders": "percent", 
                "dividends": market_currency, 
                "shares_outstanding": "shares" ,
                "identification": None
                }
    
    def price_history(self):
        return yf.download(self.ticker, interval="1mo", period="5y", auto_adjust=True, keepna=True, prepost=False)
    
    def mutualfund_holders(self):
        return self._yf.get_mutualfund_holders()

    def institutional_holders(self):
        return self._yf.get_institutional_holders()
    
    def major_holders(self):
        return self._yf.get_major_holders()

    def dividends(self):
        return self._yf.get_dividends()
    
    def shares_outstanding(self):
        return self._yf.get_shares_full()
    
    def income_statement(self):
        return self._yf.get_income_stmt()
    
    def balance_sheet(self):
        return self._yf.get_balance_sheet()

    def cashflow(self):
        return self._yf.get_cashflow()
    def get_all_yf_data(self):
        # Wir definieren, welche Methoden wir abrufen wollen
        methods = {
            #"identification": self.identification,
            "balance_sheet": self.balance_sheet,
            "income_statement": self.income_statement,
            "cashflow": self.cashflow,
            "dividends": self.dividends,
            "shares_outstanding": self.shares_outstanding,
            "mutualfund_holders": self.mutualfund_holders,
            "institutional_holders": self.institutional_holders,
            "major_holders": self.major_holders,
            "price_history": self.price_history
        }
        data_package = {}
        
        for name, method in methods.items():
            try:
                # Wir führen die Methode aus
                data = method()
                if data is not None and not data.empty:
                    data_package[name] = data
            except Exception as e:
                print(f"Fehler beim Abrufen von {name} für {self.ticker}: {e}")
                
        return data_package
    

In [4]:
class FinanceTransformer:
    @staticmethod
    def transform_identification(df_raw, ticker, affiliation):
        """Transformiert die Identifikationsdaten (ISIN)."""
        df = df_raw
        df['ticker'] = ticker
        df['affiliation'] = affiliation
        df['item_description'] = 'isin'
        df['date'] = pd.to_datetime('today').date()
        df['value'] = df.iloc[0, 0]  # Wir nehmen den ISIN-Wert aus der ersten Zelle
        
        return df[['ticker', 'date', 'affiliation', 'item_description', 'value']]
    
    @staticmethod
    def transform_fx(df_raw, base_cur, quote_cur="USD"):
        if isinstance(df_raw.columns, pd.MultiIndex):
            df_raw = df_raw.sort_index(axis=1)
            df_raw.columns = df_raw.columns.get_level_values(0)

        df = df_raw.reset_index()[["Date", "Close"]]

        df = df.rename(columns={
            "Date": "date",
            "Close": "rate"
        })

        df["base_currency"] = base_cur
        df["quote_currency"] = quote_cur

        df["date"] = pd.to_datetime(df["date"]).dt.date

        df = df[["base_currency", "quote_currency", "date", "rate"]]

        df.columns.name = None  # entfernt "Price"

        return df
    
    @staticmethod
    def transform_financial_statement(df_raw, ticker, affiliation, date_pattern = r'^\d{4}-\d{2}-\d{2}$'):
        """Transformiert Income Statement, Balance Sheet oder Cashflow.""" 
        # Das typische yfinance Format ist: Zeilen = Items, Spalten = Daten
        # Wir müssen es in das 'Long-Format' schmelzen (Melt)
        if pd.to_datetime(df_raw.columns, errors='coerce').isna().any():
            print(f"Format-Fehler: Spaltennamen von {affiliation} für {ticker} entsprechen nicht dem Datumsmuster.")
            print(df_raw.head())
            return pd.DataFrame()   
        df = df_raw.melt(ignore_index=False, var_name='date', value_name='value')
        df = df.reset_index().rename(columns={'index': 'item_description'})
        
        df['ticker'] = ticker
        df['affiliation'] = affiliation
        
        # Datentyp-Härtung
        df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None).dt.date
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        
        return df[['ticker', 'date', 'affiliation', 'item_description', 'value']]

    @staticmethod
    def transform_series(series, ticker, affiliation):           
        df = series.reset_index()
        df.columns = ['date', 'value']
        
        df['ticker'] = ticker
        df['affiliation'] = affiliation
        df['item_description'] = affiliation
        
        df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None).dt.date
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        
        return df[['ticker', 'date', 'affiliation', 'item_description', 'value']]
    
    @staticmethod
    def transform_holder_group(df_raw, ticker, affiliation):
        """Transformiert die Holder-DataFrames (Mutual, Institutional, Major)."""
        df_raw = df_raw.reset_index()
        # Wir nehmen an, dass die Holder-DataFrames eine Spalte 'Holder' und 'pctHeld' haben
        if 'Value' not in df_raw.columns or 'index' not in df_raw.columns:
            print(f"Unerwartetes Format der Eigentümer Grupen (Major Holders) von {ticker}")
            print(df_raw.head())
            return pd.DataFrame()
        
        df = df_raw.rename(columns={'index': 'item_description', 'Value': 'value'})
        df['ticker'] = ticker
        df['affiliation'] = affiliation
        df['date'] = pd.to_datetime('today').date()
        
        return df[['ticker', 'date', 'affiliation', 'item_description', 'value']]
    
    @staticmethod
    def transform_holders(df_raw, ticker, affiliation):
        """Transformiert die Holder-DataFrames (Mutual, Institutional, Major)."""       
        # Wir nehmen an, dass die Holder-DataFrames eine Spalte 'Holder' und 'pctHeld' haben
        if 'Holder' not in df_raw.columns or 'pctHeld' not in df_raw.columns:
            print(f"Unerwartetes Format für Eigentümer (Mutual Funds & Institutional) von {ticker}")
            print(df_raw.head())
            return pd.DataFrame()
        
        df = df_raw.rename(columns={'Holder': 'item_description', 'pctHeld': 'value', 'Date Reported': 'date'})
        df = df.drop(columns=["Shares", "Value", "pctChange"], errors='ignore')
        df['ticker'] = ticker
        df['affiliation'] = affiliation        
        return df[['ticker', 'date', 'affiliation', 'item_description', 'value']]
    @staticmethod
    def transform_prices(raw_data, ticker, affiliation):
        if isinstance(raw_data.columns, pd.MultiIndex):
            raw_data = raw_data.sort_index(axis=1)
            raw_data.columns = raw_data.columns.get_level_values(0)

        raw_data = raw_data.reset_index()
        value_cols = [col for col in raw_data.columns if col != "Date"]
        long_df = raw_data.melt(
        id_vars="Date",
        value_vars=value_cols,
        var_name="item_description",
        value_name="value"
        )

        long_df["ticker"] = ticker
        long_df["affiliation"] = affiliation

        long_df = long_df.rename(columns={"Date": "date"})
        long_df["date"] = pd.to_datetime(long_df["date"]).dt.date

        long_df = long_df[
            ["ticker", "date", "affiliation", "item_description", "value"]
        ].sort_values(["date", "item_description"], ignore_index=True)
        return long_df
    @staticmethod
    def insert_unit(df, unit):
        df = df.copy()
        df["unit"] = df["affiliation"].map(unit)
        df.loc[df["item_description"] == "Volume", "unit"] = "shares"
        return df


    @staticmethod
    def smart_transform(raw_data, ticker, affiliation):
        # --- FALL 1: DATAFRAMES ---
        if isinstance(raw_data, pd.DataFrame):
            # Wir prüfen sowohl Spalten als auch den Index-Namen
            cols = [str(c) for c in raw_data.columns]
            index_name = raw_data.index.name

            # A) Check für Major Holders
            if affiliation == 'major_holders':
                return FinanceTransformer.transform_holder_group(raw_data, ticker, affiliation)
            
            if affiliation == "price_history":
                return FinanceTransformer.transform_prices(raw_data, ticker, affiliation)
            
            # B) Check für Institutional/Mutualfund Holders
            elif affiliation == 'institutional_holders' or affiliation == 'mutualfund_holders':
                return FinanceTransformer.transform_holders(raw_data, ticker, affiliation)
            
            # C) Standard: Finanzberichte
            elif affiliation in ['balance_sheet', 'income_statement', 'cashflow']:
                return FinanceTransformer.transform_financial_statement(raw_data, ticker, affiliation)
            elif affiliation == 'identification':
                return FinanceTransformer.transform_identification(raw_data, ticker, affiliation)
        
        # --- FALL 2: SERIES ---
        elif isinstance(raw_data, pd.Series):
            return FinanceTransformer.transform_series(raw_data, ticker, affiliation)
        
        return pd.DataFrame()

In [5]:
from pathlib import Path
import duckdb
import pandas as pd


class DataHub:
    def __init__(self, db_name="bronze_company.db"):
        # Pfad-Management (funktioniert in Scripts & Notebooks)
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / db_name
        db_path.parent.mkdir(parents=True, exist_ok=True)

        # Verbindung herstellen
        self.con = duckdb.connect(str(db_path))
        self._initialize_tables()
        print(f"DuckDB verbunden: {db_path}")

    def _initialize_tables(self):
        """Erstellt die Tabellenstruktur, falls sie noch nicht existiert."""

        # Yahoo / Financials
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_financials (
                ticker VARCHAR,
                date DATE,
                affiliation VARCHAR,
                item_description VARCHAR,
                value DOUBLE,
                unit VARCHAR,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_ticker_date
            ON bronze_financials (ticker, date);
        """)

        # FX Rates
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_fx_rates (
                base_currency VARCHAR,
                quote_currency VARCHAR,
                date DATE,
                rate DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_fx_base_quote_date
            ON bronze_fx_rates (base_currency, quote_currency, date);
        """)

        # Wikidata
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_wikidata (
                company_qid VARCHAR,
                item_description VARCHAR,
                value VARCHAR,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid
            ON bronze_wikidata (company_qid);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid_item
            ON bronze_wikidata (company_qid, item_description);
        """)

        # GMD
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_gmd (
                countryname VARCHAR,
                iso3 VARCHAR,
                year INTEGER,
                item_description VARCHAR,
                value DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_iso3_year
            ON bronze_gmd (iso3, year);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_item
            ON bronze_gmd (item_description);
        """)

    def insert_financials(self, df: pd.DataFrame):
        """Speichert Yahoo-Financials in die Datenbank."""
        if df is None or df.empty:
            return

        df = df.copy()

        try:
            self.con.execute("""
                INSERT INTO bronze_financials (
                    ticker, date, affiliation, item_description, value, unit
                )
                SELECT
                    ticker, date, affiliation, item_description, value, unit
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_financials: {e}")

    def insert_fx_rates(self, df: pd.DataFrame, replace_existing: bool = True):
        """
        Speichert FX-Zeitreihen in bronze_fx_rates.

        Erwartete Spalten:
        - base_currency
        - quote_currency
        - date
        - rate
        """
        if df is None or df.empty:
            return

        fx_df = df.copy()

        required = {"base_currency", "quote_currency", "date", "rate"}
        if not required.issubset(fx_df.columns):
            missing = required - set(fx_df.columns)
            raise ValueError(
                f"FX DF fehlt Spalten: {missing}. Vorhanden: {list(fx_df.columns)}"
            )

        # Typen normalisieren
        fx_df["base_currency"] = fx_df["base_currency"].astype(str).str.upper()
        fx_df["quote_currency"] = fx_df["quote_currency"].astype(str).str.upper()
        fx_df["date"] = pd.to_datetime(fx_df["date"], errors="coerce").dt.date
        fx_df["rate"] = pd.to_numeric(fx_df["rate"], errors="coerce")

        # Ungültige Zeilen entfernen
        fx_df = fx_df.dropna(subset=["base_currency", "quote_currency", "date", "rate"])

        # Doppelte Zeilen im Input entfernen
        fx_df = fx_df.drop_duplicates(subset=["base_currency", "quote_currency", "date"])

        try:
            self.con.register("fx_df", fx_df)

            if replace_existing:
                self.con.execute("""
                    DELETE FROM bronze_fx_rates
                    USING fx_df
                    WHERE bronze_fx_rates.base_currency = fx_df.base_currency
                      AND bronze_fx_rates.quote_currency = fx_df.quote_currency
                      AND bronze_fx_rates.date = fx_df.date
                """)

            self.con.execute("""
                INSERT INTO bronze_fx_rates (
                    base_currency, quote_currency, date, rate
                )
                SELECT
                    base_currency,
                    quote_currency,
                    date,
                    rate
                FROM fx_df
            """)

            self.con.unregister("fx_df")

        except Exception as e:
            print(f"Fehler beim Insert in bronze_fx_rates: {e}")

    def insert_wikidata(self, df: pd.DataFrame):
        """
        Speichert Wikidata-DataFrame in die Datenbank.
        Erwartete Spalten: company / Company_QID, Item_Description, Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        if "Company_QID" not in df.columns:
            if "company" in df.columns:
                df["Company_QID"] = df["company"].astype(str)
            else:
                raise ValueError(
                    f"Wikidata DF braucht 'Company_QID' oder 'company'. Vorhanden: {list(df.columns)}"
                )

        required = {"Company_QID", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"Wikidata DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        df["Company_QID"] = df["Company_QID"].astype(str)
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = df["Value"].astype(str)

        try:
            self.con.execute("""
                INSERT INTO bronze_wikidata (
                    company_qid, item_description, value
                )
                SELECT
                    Company_QID AS company_qid,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_wikidata: {e}")

    def insert_gmd(self, df: pd.DataFrame):
        """
        Speichert GMD-Long-DataFrame in die Datenbank.

        Erwartete Spalten:
        countryname | ISO3 | year | Item_Description | Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        required = {"countryname", "ISO3", "year", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"GMD DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        df["countryname"] = df["countryname"].astype(str)
        df["ISO3"] = df["ISO3"].astype(str)
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

        df = df[df["year"].notna()].copy()
        df["year"] = df["year"].astype(int)

        try:
            self.con.execute("""
                INSERT INTO bronze_gmd (
                    countryname, iso3, year, item_description, value
                )
                SELECT
                    countryname,
                    ISO3 AS iso3,
                    year,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_gmd: {e}")

    def preview_data(self, table="bronze_financials", limit=5000):
        """Holt Einträge als DataFrame zur Kontrolle."""
        df = self.con.execute(f"SELECT * FROM {table} LIMIT {int(limit)}").df()
        return df

    def get_summary_stats(self):
        """Gibt eine kleine Statistik über den Füllstand der DB aus."""
        return self.con.execute("""
            SELECT
                (SELECT COUNT(DISTINCT ticker) FROM bronze_financials) AS count_tickers,
                (SELECT COUNT(*) FROM bronze_financials) AS total_financial_rows,
                (SELECT COUNT(*) FROM bronze_fx_rates) AS total_fx_rows,
                (SELECT COUNT(DISTINCT base_currency || '-' || quote_currency) FROM bronze_fx_rates) AS count_fx_pairs,
                (SELECT COUNT(DISTINCT company_qid) FROM bronze_wikidata) AS count_company_qids,
                (SELECT COUNT(*) FROM bronze_wikidata) AS total_wikidata_rows,
                (SELECT COUNT(DISTINCT iso3) FROM bronze_gmd) AS count_gmd_countries,
                (SELECT COUNT(*) FROM bronze_gmd) AS total_gmd_rows
        """).df()

    def close(self):
        """Schließt die Verbindung sauber."""
        self.con.close()

    def clear_specifi_database(self, table):
        try:
            self.con.execute(f"DROP TABLE IF EXISTS {table}")
            print(f"Datenbank {table} wurde erfolgreich geleert.")
        except Exception as e:
            print(f"Fehler beim Leeren der Datenbank: {e}")

    def clear_database(self):
        """Löscht alle Daten und Tabellen aus der DuckDB-Datenbank."""
        try:
            tables = self.con.execute("SHOW TABLES").fetchall()

            if not tables:
                print("Datenbank ist bereits leer.")
                return

            print(f"Lösche {len(tables)} Tabellen...")

            for (table_name,) in tables:
                self.con.execute(f"DROP TABLE IF EXISTS {table_name}")

            print("Datenbank wurde erfolgreich geleert.")

        except Exception as e:
            print(f"Fehler beim Leeren der Datenbank: {e}")

In [6]:
class YahooIngestor:
    def __init__(self, tickers: list[str]):
        self.tickers = tickers

    def get_currency(self):
        pass
    def build_currency_ticker(self, currency):
        currency_ticker = f'{currency}USD=X'
        return currency_ticker
    def clean_currency_list(self, currency_list):
        return list(set(currency_list))
    
    def currency(self, currency):
        curr_ticker = self.build_currency_ticker(currency)
        curr_timeseries = YahooClient(curr_ticker).get_fx()
        tranformed_currency_timeseries = FinanceTransformer.transform_fx(curr_timeseries, currency)
        return tranformed_currency_timeseries
        
    def financials(self, ticker):
        client = YahooClient(ticker)
        data = client.get_all_yf_data()
        unit = client.unit_info()
        market_currency = unit['price_history']
        financial_currency = unit["balance_sheet"]
        result = []
        for model_name, raw_data in data.items():
            if raw_data is None or not isinstance(raw_data, (pd.DataFrame, pd.Series)) or raw_data.empty:
                msg = f"Keine validen Daten für {model_name} bei {client.ticker} (None oder kein Pandas Objekt oder Leeres Objekt)"
                print(msg)            
                logging.warning(msg)
                continue
            clean_df = FinanceTransformer.smart_transform(raw_data, ticker=client.ticker, affiliation=model_name)
            df_units = FinanceTransformer.insert_unit(clean_df, unit)
            result.append(df_units)
        res = pd.concat(result, ignore_index=True)
        return res, market_currency, financial_currency

    def ingest(self):
        hub = DataHub()
        currencys = []
        for ticker in self.tickers:
            try:
                df, market_currency, financial_currency = self.financials(ticker)
                currencys.extend([market_currency, financial_currency])
                hub.insert_financials(df)
            except Exception as e:
                print(f"Fehler bei Ticker {ticker}: {e}")
        curr_list = self.clean_currency_list(currencys)
        for currency in curr_list:
            try:
                df = self.currency(currency)            
                hub.insert_fx_rates(df)
            except Exception as e:
                print(f'Fehler bei Currency {currency}: {e}')
        hub.close()        
        

In [7]:
tickers = pd.read_csv("C:\\Diversification\\data\\tickers_08_06_2026.csv")["YahooTicker"].tolist()
YahooIngestor(tickers).ingest()

DuckDB verbunden: c:\Diversification\data\bronze_company.db


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Fehler bei Ticker ACHR.WS: No objects to concatenate


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Fehler bei Ticker AGM.A: No objects to concatenate


...

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Fehler bei Ticker MLVDN.LS: No objects to concatenate


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Fehler bei Ticker 2955.HK: No objects to concatenate


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [9]:
DataHub().close()
len(DataHub().preview_data(table='bronze_financials', limit=500000000000))

DuckDB verbunden: c:\Diversification\data\bronze_company.db
DuckDB verbunden: c:\Diversification\data\bronze_company.db


9877572

In [10]:
df = DataHub().preview_data(table='bronze_financials', limit=750000000600)
df.head()

DuckDB verbunden: c:\Diversification\data\bronze_company.db


,ticker,date,affiliation,item_description,value,unit,ingested_at
0,AAL,2025-12-31,balance_sheet,TreasurySharesNumber,NaN,USD,2026-06-08 08:05:27.291865
1,AAL,2025-12-31,balance_sheet,OrdinarySharesNumber,6.603011e+08,USD,2026-06-08 08:05:27.291865
2,AAL,2025-12-31,balance_sheet,ShareIssued,6.603011e+08,USD,2026-06-08 08:05:27.291865
3,AAL,2025-12-31,balance_sheet,NetDebt,2.824000e+10,USD,2026-06-08 08:05:27.291865
4,AAL,2025-12-31,balance_sheet,TotalDebt,3.688400e+10,USD,2026-06-08 08:05:27.291865


In [13]:
DataHub().close()
fx = DataHub().preview_data(table='bronze_fx_rates', limit=500000000000)
print(len(fx))
DataHub().close()


DuckDB verbunden: c:\Diversification\data\bronze_company.db
DuckDB verbunden: c:\Diversification\data\bronze_company.db
148912
DuckDB verbunden: c:\Diversification\data\bronze_company.db
